In [2]:
import pandas as pd
import scipy.stats as stats
import numpy as np
import ast

In [3]:
def compute_accuracy_local(df):
    list_Trustworthiness = df["Trustworthiness"].tolist()
    list_Continuity = df["Continuity"].tolist()

    results = []
    for i in range(len(list_Trustworthiness)):
        results.append(round(0.5*list_Trustworthiness[i] + 0.5*list_Continuity[i], 2))
    return results

In [4]:
def compute_accuracy_global(df):
    list_Shephard = df["Shephard Diagram Correlation"].tolist()

    results = []
    for i in range(len(list_Shephard)):
        results.append(round(0.5*(list_Shephard[i] + 1), 2))
    return results

In [5]:
def compute_perception(df):
    list_NeighborhoodHit = df["7-Neighborhood Hit"].tolist()
    list_DistanceConsistency = df["Distance consistency"].tolist()

    results = []
    for i in range(len(list_NeighborhoodHit)):
        results.append(round(0.5*list_NeighborhoodHit[i] + 0.5*list_DistanceConsistency[i], 2))
    return results

In [6]:
def extract_K(df, K):
    # Define excluded embeddings
    excluded_embeddings = ['bert', 'bow', 'tfidf']

    # Keep only TMs (LDA, LSI, NMF) and make a copy to avoid SettingWithCopyWarning
    df_TMs = df[~df['TM'].isin(excluded_embeddings)].copy()

    # Extract the number of topics from the experiment name
    df_TMs['n_topics'] = df_TMs['Experiment'].str.extract(r'n_topics_(\d+)_')[0]

    # Keep only rows with the desired number of topics
    df_TMs = df_TMs[df_TMs['n_topics'] == str(K)]

    # Keep rows with no TMs and make a copy
    df_noTMs = df[df['TM'].isin(excluded_embeddings)].copy()
    df_noTMs['n_topics'] = str(K)

    # Combine the two DataFrames
    df_result = pd.concat([df_TMs, df_noTMs], ignore_index=True)

    return df_result


In [7]:
def process_df(file, corpus, K):
    df = pd.read_csv(file)
    df["corpus"] = corpus
    df["accuracy_local"] = compute_accuracy_local(df)
    df["accuracy_global"] = compute_accuracy_global(df)
    df["perception"] = compute_perception(df)
    df_result = extract_K(df, K)
    return df_result

In [8]:
df_20Newsgroups = process_df("results_cluster/cur_res/full_res_20_newsgroups.csv", "20Newsgroups", K = 20)
print(set(df_20Newsgroups["DR"].tolist()))
print(len(set(df_20Newsgroups["TM"].tolist())))
print(df_20Newsgroups.shape)

{'umap', 'tsne', 'som'}
13
(3767, 19)


In [9]:
df_20Newsgroups.head(7)

,Experiment,Trustworthiness,Continuity,Shephard Diagram Correlation,Normalized Stress,7-Neighborhood Hit,Calinski-Harabasz-Index,Silhouette coefficient,Davies-Bouldin-Index,SDBW validity index,Distance consistency,Complete List of Hyperparameters,DR,TM,corpus,accuracy_local,accuracy_global,perception,n_topics
0,20_newsgroups_lda_linear_combined_n_topics_20_...,0.974332,0.916557,0.275004,1405.133478,0.234109,405.447207,-0.252267,27.231075,1.635301,0.117112,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.95,0.64,0.18,20
1,20_newsgroups_lda_linear_combined_n_topics_20_...,0.985771,0.948061,0.466333,1654.512319,0.259198,587.489561,-0.158022,10.992132,0.859916,0.253049,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.97,0.73,0.26,20
2,20_newsgroups_lda_linear_combined_n_topics_20_...,0.989056,0.948534,0.445717,2859.867981,0.287735,545.010091,-0.167776,11.745400,0.894751,0.256231,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.97,0.72,0.27,20
3,20_newsgroups_lda_linear_combined_n_topics_20_...,0.986941,0.943132,0.420994,1707.888333,0.265070,486.487251,-0.157762,14.491598,0.938828,0.244653,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.97,0.71,0.25,20
4,20_newsgroups_lda_linear_combined_n_topics_20_...,0.979148,0.945763,0.518933,812.048368,0.225789,644.093971,-0.162864,11.148190,0.918531,0.249867,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.96,0.76,0.24,20
5,20_newsgroups_lda_linear_combined_n_topics_20_...,0.985401,0.941952,0.392818,1801.257238,0.256736,510.832266,-0.187291,13.026445,0.870767,0.242355,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.96,0.70,0.25,20
6,20_newsgroups_lda_linear_combined_n_topics_20_...,0.981774,0.945589,0.490782,1141.106742,0.245789,540.138191,-0.159737,13.327673,0.948599,0.233339,"{'lda': {'n_topics': 20, 'alpha': 'auto', 'ite...",som,lda_linear_combined,20Newsgroups,0.96,0.75,0.24,20


In [10]:
df_Emails = process_df("results_cluster/cur_res/full_res_emails.csv", "Emails", K = 8)
print(set(df_Emails["DR"].tolist()))
print(len(set(df_Emails["TM"].tolist())))
print(df_Emails.shape)

{'som', 'tsne', 'umap', 'mds'}
12
(3622, 19)


In [11]:
df_BBC = process_df("results_cluster/cur_res/full_res_bbc_news.csv", "BBC", K = 10)
print(set(df_BBC["DR"].tolist()))
print(len(set(df_BBC["TM"].tolist())))
print(df_BBC.shape)

{'som', 'tsne', 'umap', 'mds'}
13
(3834, 19)


In [12]:
df_lyrics = process_df("results_cluster/cur_res/full_res_lyrics.csv", "Lyrics", K = 8)
print(set(df_lyrics["DR"].tolist()))
print(len(set(df_lyrics["TM"].tolist())))
print(df_lyrics.shape)

{'som', 'tsne', 'umap', 'mds'}
13
(3849, 19)


In [13]:
df_Reuters = process_df("results_cluster/cur_res/full_res_reuters.csv", "Reuters", K = 10)
print(set(df_Reuters["DR"].tolist()))
print(len(set(df_Reuters["TM"].tolist())))
print(df_Reuters.shape)

{'som', 'tsne', 'umap', 'mds'}
13
(3849, 19)


In [14]:
df_7Categories = process_df("results_cluster/cur_res/full_res_seven_categories.csv", "7Categories", K = 14)
print(set(df_7Categories["DR"].tolist()))
print(len(set(df_7Categories["TM"].tolist())))
print(df_7Categories.shape)

{'umap', 'tsne', 'som'}
13
(3754, 19)


In [15]:
df_all_corpora = pd.concat([df_20Newsgroups, df_Emails, df_BBC, df_lyrics, df_Reuters, df_7Categories], ignore_index=True)
df_all_corpora.shape

(22675, 19)

In [16]:
#DR_list = ['mds', 'som', 'tsne', 'umap']
#TM_list = ['bow','tfidf','lda','lda_linear_combined','lsi','lsi_linear_combined','lsi_tfidf','lsi_tfidf_linear_combined','nmf','nmf_linear_combined',
#           'nmf_tfidf','nmf_tfidf_linear_combined','bert']

#for DR in DR_list:
#    for TM in TM_list:
#        name = "analysis-results/boxplot-data/" + DR + "-" + TM + ".csv"
#        df_selected = df_all_corpora[(df_all_corpora["DR"] == DR) & (df_all_corpora["TM"] == TM)]
#        df_selected_short = df_selected[["DR", "TM", "accuracy_local", "accuracy_global", "perception"]]
#        df_selected_short.to_csv(name)

In [17]:
DR_list = ['mds', 'som', 'tsne', 'umap']
TM_list = ['bow','tfidf','lda','lda_linear_combined','lsi','lsi_linear_combined','lsi_tfidf','lsi_tfidf_linear_combined','nmf','nmf_linear_combined',
           'nmf_tfidf','nmf_tfidf_linear_combined','bert']

for DR in DR_list:
    for TM in TM_list:
        name = "analysis-results/boxplot-data-ueberpruefung/" + DR + "-" + TM + ".csv"
        df_selected = df_all_corpora[(df_all_corpora["DR"] == DR) & (df_all_corpora["TM"] == TM)]
        df_selected_short = df_selected[["DR", "TM", "accuracy_local", "accuracy_global", "perception"]]
        df_selected_short.to_csv(name)